# YOLO26 Training — Coca Cola Bottle Dataset (Local)

**Model:** YOLO26 (Ultralytics ≥ 8.3.x)  
**Runtime:** Local machine — GPU strongly recommended (CUDA)  

> **YOLO26 highlights:** NMS-free end-to-end inference, no DFL, MuSGD optimizer,
> ProgLoss + STAL label assignment. Up to 43 % faster CPU inference vs YOLO11.

### Expected dataset layout
```
datasets/cocacola/
├── data.yaml
├── train/
│   ├── images/
│   └── labels/
├── valid/          # or val/
│   ├── images/
│   └── labels/
└── test/           # optional
    ├── images/
    └── labels/
```
Place your Roboflow / custom YOLO-format dataset in that folder before running.

---

## 1. Install Ultralytics

In [ ]:
# YOLO26 requires ultralytics >= 8.3.x
# Run once; skip if already installed.
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "ultralytics", "-q"])

import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")
ultralytics.checks()  # Verifies GPU / CUDA / library versions

## 2. Configure Paths

In [ ]:
import os
import yaml

# ── ✏️  EDIT THESE to match your local setup ─────────────────────────────
DATASET_DIR = os.path.join(os.getcwd(), "datasets", "cocacola")  # path to your dataset root
RUNS_DIR    = os.path.join(os.getcwd(), "runs")                   # where training outputs are saved
# ─────────────────────────────────────────────────────────────────────────

YAML_PATH = os.path.join(DATASET_DIR, "data.yaml")

# Sanity checks
assert os.path.isdir(DATASET_DIR), (
    f"Dataset folder not found: {DATASET_DIR}\n"
    "Please set DATASET_DIR to the folder that contains data.yaml."
)
assert os.path.isfile(YAML_PATH), (
    f"data.yaml not found inside {DATASET_DIR}\n"
    "Make sure your Roboflow / custom export includes data.yaml."
)

print("Dataset contents:")
for item in sorted(os.listdir(DATASET_DIR)):
    print(f"  {item}")
print(f"\nYAML path : {YAML_PATH}")
print(f"Runs dir  : {RUNS_DIR}")

## 3. Patch data.yaml for Local Paths

In [ ]:
# Make sure data.yaml points to this machine's absolute paths.
with open(YAML_PATH, "r") as f:
    data_cfg = yaml.safe_load(f)

print("Original data.yaml:")
print(yaml.dump(data_cfg))

# Overwrite 'path' with the absolute local dataset root
data_cfg["path"] = os.path.abspath(DATASET_DIR)

# Auto-detect train / val / test splits if missing
for split, candidates in {
    "train": ["train/images", "train"],
    "val":   ["valid/images", "val/images", "valid", "val"],
    "test":  ["test/images",  "test"],
}.items():
    if split not in data_cfg:
        for c in candidates:
            if os.path.exists(os.path.join(DATASET_DIR, c)):
                data_cfg[split] = c
                print(f"  Auto-detected {split}: {c}")
                break

with open(YAML_PATH, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print("\nPatched data.yaml:")
print(yaml.dump(data_cfg))

## 4. Detect Device (GPU / CPU)

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = 0  # first CUDA GPU
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif torch.backends.mps.is_available():
    DEVICE = "mps"  # Apple Silicon
    print("✅ Apple MPS detected")
else:
    DEVICE = "cpu"
    print("⚠️  No GPU found — training on CPU will be very slow.")
    print("   Consider reducing epochs and image size for a quick test.")

# Auto-scale batch size: reduce if running on CPU or low VRAM
BATCH = 16 if DEVICE != "cpu" else 4
print(f"\nDevice: {DEVICE} | Batch size: {BATCH}")

## 5. Configure Training Parameters

In [ ]:
# ── YOLO26 model variants ─────────────────────────────────────────────────
#   yolo26n.pt  — nano   (2.4 M params, fastest, ~40.9 mAP)  ← good for CPU tests
#   yolo26s.pt  — small  (9.5 M params, ~48.6 mAP)
#   yolo26m.pt  — medium (20.4 M params, ~53.1 mAP)          ← balanced default
#   yolo26l.pt  — large  (~54.3 mAP)
#   yolo26x.pt  — xlarge (~56.8 mAP, slowest)

TRAIN_CONFIG = dict(
    model         = "yolo26m.pt",   # ✏️ change variant here
    data          = YAML_PATH,
    epochs        = 200,
    imgsz         = 640,
    batch         = BATCH,
    # patience    = 80,             # uncomment to enable early stopping
    optimizer     = "MuSGD",
    lr0           = 1e-2,
    lrf           = 0.01,
    momentum      = 0.937,
    weight_decay  = 5e-4,
    warmup_epochs = 3,
    cos_lr        = True,
    augment       = True,
    degrees       = 5.0,
    flipud        = 0.0,
    fliplr        = 0.0,            # bottles have orientation — no horizontal flip
    hsv_h         = 0.015,
    hsv_s         = 0.4,
    hsv_v         = 0.4,
    mosaic        = 1.0,
    mixup         = 0.0,
    copy_paste    = 0.0,
    workers       = 4,
    device        = DEVICE,
    project       = RUNS_DIR,
    name          = "cocacola_yolo26m_local_v1",
    exist_ok      = True,
    pretrained    = True,
    verbose       = True,
)

print("Training config:")
for k, v in TRAIN_CONFIG.items():
    print(f"  {k:20s}: {v}")

## 6. Train the Model

In [ ]:
from ultralytics import YOLO

os.makedirs(RUNS_DIR, exist_ok=True)

model = YOLO(TRAIN_CONFIG.pop("model"))
results = model.train(**TRAIN_CONFIG)

SAVE_DIR = str(results.save_dir)
print("\n✅ Training complete!")
print(f"Best weights : {SAVE_DIR}/weights/best.pt")
print(f"Last weights : {SAVE_DIR}/weights/last.pt")

## 7. Evaluate on Validation Set

In [ ]:
best_weights = os.path.join(SAVE_DIR, "weights", "best.pt")
eval_model   = YOLO(best_weights)

metrics = eval_model.val(data=YAML_PATH, imgsz=640, batch=BATCH, device=DEVICE)

print("\n📊 Validation metrics:")
print(f"  mAP50      : {metrics.box.map50:.4f}")
print(f"  mAP50-95   : {metrics.box.map:.4f}")
print(f"  Precision  : {metrics.box.mp:.4f}")
print(f"  Recall     : {metrics.box.mr:.4f}")

## 8. Plot Training Curves

In [ ]:
from IPython.display import Image as IPyImage, display
import glob

for plot_file in ["results.png", "confusion_matrix.png", "PR_curve.png", "F1_curve.png"]:
    path = os.path.join(SAVE_DIR, plot_file)
    if os.path.exists(path):
        print(f"\n📈 {plot_file}")
        display(IPyImage(path))
    else:
        print(f"  (not found: {plot_file})")

## 9. Run Inference on Test Images

In [ ]:
# Try test/, then valid/ as fallback
for split in ["test", "valid", "val"]:
    test_images_dir = os.path.join(DATASET_DIR, split, "images")
    if os.path.exists(test_images_dir):
        break
else:
    test_images_dir = None

if test_images_dir:
    test_imgs = glob.glob(os.path.join(test_images_dir, "*.jpg"))[:5]
    if not test_imgs:  # also try PNG
        test_imgs = glob.glob(os.path.join(test_images_dir, "*.png"))[:5]

    if test_imgs:
        pred_results = eval_model.predict(
            source   = test_imgs,
            imgsz    = 640,
            conf     = 0.25,
            iou      = 0.45,
            save     = True,
            project  = RUNS_DIR,
            name     = "cocacola_predictions",
            exist_ok = True,
            device   = DEVICE,
        )
        pred_dir = str(pred_results[0].save_dir)
        for img_path in sorted(glob.glob(os.path.join(pred_dir, "*.jpg"))):
            print(f"\n🖼  {os.path.basename(img_path)}")
            display(IPyImage(img_path))
    else:
        print(f"No images found in {test_images_dir}")
else:
    print("No test/valid/val images folder found in dataset.")

## 10. Export Model (optional)

In [ ]:
# Export to ONNX for deployment (CPU-friendly, framework-agnostic)
# Other formats: 'torchscript', 'tflite', 'coreml', 'engine' (TensorRT)

export_model = YOLO(best_weights)
export_path  = export_model.export(format="onnx", imgsz=640, dynamic=True)
print(f"\n📦 Exported ONNX model: {export_path}")

---
## Summary

| Item | Location |
|------|----------|
| Best weights (`.pt`) | `runs/cocacola_yolo26m_local_v1/weights/best.pt` |
| Last weights (`.pt`) | `runs/cocacola_yolo26m_local_v1/weights/last.pt` |
| Training curves | `runs/cocacola_yolo26m_local_v1/results.png` |
| Predictions | `runs/cocacola_predictions/` |
| ONNX export | next to `best.pt` in the weights folder |

> All paths above are relative to the directory where you launched Jupyter.
